# Speaker Labelling

After diarization, we wish to post-process the transcription text with its labels: 'Speaker-X' where X={1,2,3...} or 'Speaker-Unknown' in the case where the model fails to assign a speaker to the model.

We demonstrate ChatGPT's ability to label the speaker according to the following classes: "Doctor", "Patient", "Caregiver", "Nurse", "Family Member", "Others". 

In [1]:
import os
from dotenv import load_dotenv
from openai import AzureOpenAI

# Set up Azure OpenAI client

load_dotenv()
AZURE_OPENAI_KEY = os.getenv("AZURE_OPENAI_KEY")
AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")

openai_client = AzureOpenAI(
    api_version="2024-02-01", # Make sure to use the correct API version
    api_key=AZURE_OPENAI_KEY,
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
)

In [29]:
def label_speaker(text):
    messages = [
        {
            "role": "system",
            "content": "You are an assistant that generates labels for speakers in transcripts of medical consultations. Your task is to analyze the text and assign a relevant role to each speaker."

        },
        {
            "role": "user",
            "content": (
                f"You are given a transcript of a medical consultation where each line is attributed to either a numbered speaker (e.g., 'Speaker-1', 'Speaker-2') or 'Speaker-Unknown': {text}"
                "Your task if to assign specific roles (e.g. 'Doctor', 'Patient', 'Caregiver', etc.) to each speaker based on the content of their dialogue."
                "If 'Speaker-Unknown' appears, infer that this speaker is one of the numbered speakers that has already appeared in the transcript."
                "In some cases, 'Speaker-Unknown' might appear multiple times and can represent multiple speakers."
                "Additionally, if a line seems irrelevant to the medical consultation or involves someone outside of the consultation (e.g., a random person asking an unrelated question), assign the role 'Others.'"
                
                "Here are the possible roles that can appear in a medical consultation:"
                " - Doctor: Medical professional providing advice, diagnosis or treatment."
                " - Patient: Individual seeking medical attention."
                " - Caregiver: A person accompanying the patient, providing support."
                " - Nurse: A medical professional assisting with patient care."
                " - Family Member: A relative of the patient, offering support."
                " - Others: A person speaking in the conversation but not directly related to the medical consultation (e.g., someone passing by or asking an unrelated question)."

                "The output should look something like this:"
                "Doctor: Good morning, how are you feeling today?"
                "Patient: I've been feeling dizzy for the last two days."
                "Doctor: Can you describe the dizziness in more detail?"
                "Speaker Unknown (Family Member): I also noticed the patient had a fever last night."
                "Caregiver: Yes, that’s right. It started in the evening."
                "Speaker Unknown (Family Member): The symptoms seem to be getting worse."
            )
        }

    ]

    # Request to the Azure OpenAI API using chat completions
    completion = openai_client.chat.completions.create(
        model="gpt-4o-mini",  # Replace with your Azure GPT model deployment name
        messages=messages,
        #max_tokens=200,  # Limit tokens for a concise response
        temperature=0.7,  # Adjust to control creativity level
        n=1,  # Number of responses
    )

    # Extract and return the generated titles
    output = completion.choices[0].message.content.strip()
    return output

## Case 1: "Speaker-Unknown" is 1 Speaker

In [30]:
example_transcript1 = """
Speaker-1: Hey, good afternoon, Mr/Ms [Name]. My name is Dr [Name]. Nice to meet you. So I understand that you've been having some weight loss and you've been feeling hot all the time. Could you tell me a bit more about that?
Speaker-2: Yes. And on top of that, I also feel like I'm having a lot of, like, complications, like having very fast heart rate and I'm hungry all the time. But I'm not gaining any weight… I'm losing weight.
Speaker-1: I see. And are you having any diarrhoea?
Speaker-2: Uh, yes, for the last two months.
Speaker-1: Got it. Do you have any past medical history I should know about?
Speaker-2: I have diabetes and high blood pressure.
Speaker-Unknown: Diabetes and high blood pressure. Are you taking any medications for that?
Speaker-2: Yeah, I'm taking, uh, metformin and amlodipine.
Speaker-1: I see. And do you have any family members who also, uh, you know, got health conditions, for example, thyroid disease?
Speaker-2: Uh, actually can I speak in Mandarin 'cause I'm a bit more comfortable with that?
Speaker-1: Sure.
Speaker-2: 所以我的家庭...
Speaker-2: 啊，我的妈妈有啊，甲状腺的问题，她的荷尔蒙激素比较高。
Speaker-Unknown: 你的母亲有甲状腺的问题，那可能有关系。明白了，我可以给你检查一下吗？
Speaker-2: 啊，可以呀。
Speaker-1: 好，可以把双手伸出来吗？
Speaker-1: 啊，看起来你是有手发抖，你的心跳也有差不多一百。嗯，我怀疑你是有高额甲状腺荷尔蒙的问题。
Speaker-Unknown: 今天我会帮你验一些血，看甲状腺的指数，明天再回来吧，可能要给你吃降甲状腺荷尔蒙的药。
Speaker-2: 好，谢谢医生。
Speaker-1: 谢谢。
"""

In [31]:
%%time

example_output1 = label_speaker(example_transcript1)
print(example_output1)

Doctor: Hey, good afternoon, Mr/Ms [Name]. My name is Dr [Name]. Nice to meet you. So I understand that you've been having some weight loss and you've been feeling hot all the time. Could you tell me a bit more about that?  
Patient: Yes. And on top of that, I also feel like I'm having a lot of, like, complications, like having very fast heart rate and I'm hungry all the time. But I'm not gaining any weight… I'm losing weight.  
Doctor: I see. And are you having any diarrhoea?  
Patient: Uh, yes, for the last two months.  
Doctor: Got it. Do you have any past medical history I should know about?  
Patient: I have diabetes and high blood pressure.  
Speaker Unknown (Doctor): Diabetes and high blood pressure. Are you taking any medications for that?  
Patient: Yeah, I'm taking, uh, metformin and amlodipine.  
Doctor: I see. And do you have any family members who also, uh, you know, got health conditions, for example, thyroid disease?  
Patient: Uh, actually can I speak in Mandarin 'cause

## Case 2: "Speaker-Unknown" is >1 Speaker

In [32]:
example_transcript2 = """
Speaker-1: Hey, good afternoon, Mr/Ms [Name]. My name is Dr [Name]. Nice to meet you. So I understand that you've been having some weight loss and you've been feeling hot all the time. Could you tell me a bit more about that?
Speaker-2: Yes. And on top of that, I also feel like I'm having a lot of, like, complications, like having very fast heart rate and I'm hungry all the time. But I'm not gaining any weight… I'm losing weight.
Speaker-1: I see. And are you having any diarrhoea?
Speaker-2: Uh, yes, for the last two months.
Speaker-1: Got it. Do you have any past medical history I should know about?
Speaker-2: I have diabetes and high blood pressure.
Speaker-Unknown: Diabetes and high blood pressure. Are you taking any medications for that?
Speaker-2: Yeah, I'm taking, uh, metformin and amlodipine.
Speaker-1: I see. And do you have any family members who also, uh, you know, got health conditions, for example, thyroid disease?
Speaker-2: Uh, actually can I speak in Mandarin 'cause I'm a bit more comfortable with that?
Speaker-1: Sure.
Speaker-2: 所以我的家庭...
Speaker-2: 啊，我的妈妈有啊，甲状腺的问题，她的荷尔蒙激素比较高。
Speaker-Unknown: 你的母亲有甲状腺的问题，那可能有关系。明白了，我可以给你检查一下吗？
Speaker-Unknown: 啊，可以呀。
Speaker-1: 好，可以把双手伸出来吗？
Speaker-1: 啊，看起来你是有手发抖，你的心跳也有差不多一百。嗯，我怀疑你是有高额甲状腺荷尔蒙的问题。
Speaker-1: 今天我会帮你验一些血，看甲状腺的指数，明天再回来吧，可能要给你吃降甲状腺荷尔蒙的药。
Speaker-2: 好，谢谢医生。
Speaker-1: 谢谢。
"""

In [33]:
%%time

example_output2 = label_speaker(example_transcript2)
print(example_output2)

Doctor: Hey, good afternoon, Mr/Ms [Name]. My name is Dr [Name]. Nice to meet you. So I understand that you've been having some weight loss and you've been feeling hot all the time. Could you tell me a bit more about that?  
Patient: Yes. And on top of that, I also feel like I'm having a lot of, like, complications, like having very fast heart rate and I'm hungry all the time. But I'm not gaining any weight… I'm losing weight.  
Doctor: I see. And are you having any diarrhoea?  
Patient: Uh, yes, for the last two months.  
Doctor: Got it. Do you have any past medical history I should know about?  
Patient: I have diabetes and high blood pressure.  
Speaker Unknown (Doctor): Diabetes and high blood pressure. Are you taking any medications for that?  
Patient: Yeah, I'm taking, uh, metformin and amlodipine.  
Doctor: I see. And do you have any family members who also, uh, you know, got health conditions, for example, thyroid disease?  
Patient: Uh, actually can I speak in Mandarin 'cause

## Case 3: Unrelated Unidentified Speaker (Noise)

In [34]:
example_transcript3 = """
Speaker-1: Hey, good afternoon, Mr/Ms [Name]. My name is Dr [Name]. Nice to meet you. So I understand that you've been having some weight loss and you've been feeling hot all the time. Could you tell me a bit more about that?
Speaker-2: Yes. And on top of that, I also feel like I'm having a lot of, like, complications, like having very fast heart rate and I'm hungry all the time. But I'm not gaining any weight… I'm losing weight.
Speaker-1: I see. And are you having any diarrhoea?
Speaker-2: Uh, yes, for the last two months.
Speaker-1: Got it. Do you have any past medical history I should know about?
Speaker-2: I have diabetes and high blood pressure.
Speaker-Unknown: Diabetes and high blood pressure. Are you taking any medications for that?
Speaker-2: Yeah, I'm taking, uh, metformin and amlodipine.
Speaker-1: I see. And do you have any family members who also, uh, you know, got health conditions, for example, thyroid disease?
Speaker-2: Uh, actually can I speak in Mandarin 'cause I'm a bit more comfortable with that?
Speaker-Unknown: Hey, do you know where to find the toilet?
Speaker-1: Sure.
Speaker-2: 所以我的家庭...
Speaker-2: 啊，我的妈妈有啊，甲状腺的问题，她的荷尔蒙激素比较高。
Speaker-Unknown: 你的母亲有甲状腺的问题，那可能有关系。明白了，我可以给你检查一下吗？
Speaker-Unknown: 啊，可以呀。
Speaker-1: 好，可以把双手伸出来吗？
Speaker-1: 啊，看起来你是有手发抖，你的心跳也有差不多一百。嗯，我怀疑你是有高额甲状腺荷尔蒙的问题。
Speaker-1: 今天我会帮你验一些血，看甲状腺的指数，明天再回来吧，可能要给你吃降甲状腺荷尔蒙的药。
Speaker-2: 好，谢谢医生。
Speaker-1: 谢谢。
"""

In [35]:
%%time

example_output3 = label_speaker(example_transcript3)
print(example_output3)

Doctor: Hey, good afternoon, Mr/Ms [Name]. My name is Dr [Name]. Nice to meet you. So I understand that you've been having some weight loss and you've been feeling hot all the time. Could you tell me a bit more about that?  
Patient: Yes. And on top of that, I also feel like I'm having a lot of, like, complications, like having very fast heart rate and I'm hungry all the time. But I'm not gaining any weight… I'm losing weight.  
Doctor: I see. And are you having any diarrhoea?  
Patient: Uh, yes, for the last two months.  
Doctor: Got it. Do you have any past medical history I should know about?  
Patient: I have diabetes and high blood pressure.  
Speaker Unknown (Doctor): Diabetes and high blood pressure. Are you taking any medications for that?  
Patient: Yeah, I'm taking, uh, metformin and amlodipine.  
Doctor: I see. And do you have any family members who also, uh, you know, got health conditions, for example, thyroid disease?  
Patient: Uh, actually can I speak in Mandarin 'cause